### Imports & Setup

Import all necessary libraries and custom functions:

In [1]:
%pip install -r requirements.txt

import os, sys, subprocess
import zipfile
import pandas as pd
import pm4py
from logview.utils import LogViewBuilder
from logview.predicate import *
from filter_visualization_2 import query_exploration_icicle, query_breakdown_pie
from pm4py.objects.log.importer.xes import importer as xes_importer
from pm4py.algo.discovery.dfg import algorithm as dfg_discovery
from pm4py.visualization.dfg import visualizer as dfg_visualizer

You should consider upgrading via the 'c:\Program Files\Python310\python.exe -m pip install --upgrade pip' command.


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
if not os.path.exists("logview"):
    subprocess.run(["git", "clone", "https://github.com/fzerbato/logview.git"], check=True)
    print("Cloned logview.")
else:
    print("logview already cloned.")

logview_path = os.path.abspath("logview")

if logview_path not in sys.path:
    sys.path.append(logview_path)
    print(f"Added to sys.path: {logview_path}")

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "logview/requirements.txt"], check=True)


logview already cloned.
Added to sys.path: c:\Users\cshek\OneDrive\Bureaublad\Master-Thesis-Experiment\logview


CompletedProcess(args=['c:\\Program Files\\Python310\\python.exe', '-m', 'pip', 'install', '-r', 'logview/requirements.txt'], returncode=0)

### Load and Prepare Event Log

Then load your event log and format it for analysis with PM4Py. The code below does this for the BPI Challenge 2017 dataset:


In [3]:
# Load data

csv_file = "BPI_Challenge_2017.csv"
zip_file = "BPI_Challenge_2017.zip"

if not os.path.exists(csv_file):
    if os.path.exists(zip_file):
        print(f"Extracting {csv_file} from {zip_file}...")
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extract(csv_file)
    else:
        raise FileNotFoundError(f"Both '{csv_file}' and '{zip_file}' not found")
    
CASE_ID_COL = 'case'
TIMESTAMP_COL = 'time'
ACTIVITY_COL = 'event'
    
bpi_data = pd.read_csv(csv_file, sep=',', quotechar='"')
bpi_data.columns = bpi_data.columns.str.strip()
bpi_data[TIMESTAMP_COL] = pd.to_datetime(bpi_data[TIMESTAMP_COL], format='%Y/%m/%d %H:%M:%S.%f')
log = pm4py.format_dataframe(bpi_data, case_id=CASE_ID_COL, activity_key=ACTIVITY_COL, timestamp_key=TIMESTAMP_COL)

display(log)

,case,event,time,lifecycle:transition,ApplicationType,LoanGoal,RequestedAmount,MonthlyCost,org:resource,Selected,...,Accepted,CreditScore,NumberOfTerms,EventOrigin,OfferedAmount,case:concept:name,concept:name,time:timestamp,@@index,@@case_index
0,Application_1000086665,A_Create Application,2016-08-03 17:57:21.673000+00:00,COMPLETE,New credit,"Other, see explanation",5000.0,NaN,User_1,NaN,...,NaN,NaN,NaN,Application,NaN,Application_1000086665,A_Create Application,2016-08-03 17:57:21.673000+00:00,0,0
1,Application_1000086665,A_Submitted,2016-08-03 17:57:21.734000+00:00,COMPLETE,New credit,"Other, see explanation",5000.0,NaN,User_1,NaN,...,NaN,NaN,NaN,Application,NaN,Application_1000086665,A_Submitted,2016-08-03 17:57:21.734000+00:00,1,0
2,Application_1000086665,W_Handle leads,2016-08-03 17:57:21.963000+00:00,SCHEDULE,New credit,"Other, see explanation",5000.0,NaN,User_1,NaN,...,NaN,NaN,NaN,Workflow,NaN,Application_1000086665,W_Handle leads,2016-08-03 17:57:21.963000+00:00,2,0
3,Application_1000086665,W_Handle leads,2016-08-03 17:58:28.286000+00:00,WITHDRAW,New credit,"Other, see explanation",5000.0,NaN,User_1,NaN,...,NaN,NaN,NaN,Workflow,NaN,Application_1000086665,W_Handle leads,2016-08-03 17:58:28.286000+00:00,3,0
4,Application_1000086665,W_Complete application,2016-08-03 17:58:28.293000+00:00,SCHEDULE,New credit,"Other, see explanation",5000.0,NaN,User_1,NaN,...,NaN,NaN,NaN,Workflow,NaN,Application_1000086665,W_Complete application,2016-08-03 17:58:28.293000+00:00,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1202262,Application_999993812,W_Call incomplete files,2016-10-20 10:19:28.812000+00:00,RESUME,New credit,Caravan / Camper,30000.0,NaN,User_41,NaN,...,NaN,NaN,NaN,Workflow,NaN,Application_999993812,W_Call incomplete files,2016-10-20 10:19:28.812000+00:00,1202262,31508
1202263,Application_999993812,W_Call incomplete files,2016-10-20 10:21:59.667000+00:00,SUSPEND,New credit,Caravan / Camper,30000.0,NaN,User_41,NaN,...,NaN,NaN,NaN,Workflow,NaN,Application_999993812,W_Call incomplete files,2016-10-20 10:21:59.667000+00:00,1202263,31508
1202264,Application_999993812,O_Accepted,2016-10-24 08:24:30.056000+00:00,COMPLETE,New credit,Caravan / Camper,30000.0,NaN,User_68,NaN,...,NaN,NaN,NaN,Offer,NaN,Application_999993812,O_Accepted,2016-10-24 08:24:30.056000+00:00,1202264,31508
1202265,Application_999993812,A_Pending,2016-10-24 08:24:30.059000+00:00,COMPLETE,New credit,Caravan / Camper,30000.0,NaN,User_68,NaN,...,NaN,NaN,NaN,Application,NaN,Application_999993812,A_Pending,2016-10-24 08:24:30.059000+00:00,1202265,31508


### Instantiating a LogView object ###
The LogViewBuilder class allows instantiating a LogView object, which serves as the central interface for creating LogView instances. It provides a single point of access for interacting with the different framework components.

In [4]:
# Build LogView
log_view = LogViewBuilder.build_log_view(log)

In [5]:
# CreditScore ≥ 600
query_1 = Query('GoodCredit', [GreaterEqualToConstant('CreditScore', 600)])
result_set_1, complement_1 = log_view.evaluate_query('rs_GoodCredit', log, query_1)

# RequestedAmount ≥ 10000
query_2 = Query('LoanOverThreshold', [GreaterEqualToConstant('RequestedAmount', 10000)])
result_set_2, complement_2 = log_view.evaluate_query('rs_LoanOverThreshold', result_set_1, query_2)

# RequestedAmount < 15000
query_3 = Query('SmallAmount', [LessThanConstant('RequestedAmount', 15000)])
result_set_3, complement_3 = log_view.evaluate_query('rs_SmallAmount', result_set_2, query_3)

# ApplicationType = 'New credit'
query_4 = Query('IsNewCredit', [EqToConstant('ApplicationType', 'New credit')])
result_set_4, complement_4 = log_view.evaluate_query('rs_IsNewCredit', result_set_3, query_4)

In [6]:
# It is possible to start another analysis on the source log, this wil also be added to the same LogView object, but can be used seperately later on:

# Starts with A_Create Application
query_5 = Query('StartWithCreate', [StartWith(['A_Create Application'])])
result_set_5, complement_5 = log_view.evaluate_query('rs_StartWithCreate', log, query_5)

# Duration between 2 and 7 days
query_6 = Query('ModerateDuration', [DurationWithin(172800, 604800)])
result_set_6, complement_6 = log_view.evaluate_query('rs_ModerateDuration', result_set_5, query_6)

In [7]:
summary = log_view.get_summary()

+----+----------------------+-------------------+----------------------+----------+
|    | source_log           | query             | result_set           | labels   |
|----+----------------------+-------------------+----------------------+----------|
|  0 | initial_source_log   | GoodCredit        | rs_GoodCredit        | []       |
|  1 | rs_GoodCredit        | LoanOverThreshold | rs_LoanOverThreshold | []       |
|  2 | rs_LoanOverThreshold | SmallAmount       | rs_SmallAmount       | []       |
|  3 | rs_SmallAmount       | IsNewCredit       | rs_IsNewCredit       | []       |
|  4 | initial_source_log   | StartWithCreate   | rs_StartWithCreate   | []       |
|  5 | rs_StartWithCreate   | ModerateDuration  | rs_ModerateDuration  | []       |
+----+----------------------+-------------------+----------------------+----------+
+----+-------------------+----------------------------------------+
|    | query             | predicates                             |
|----+------------------

In [8]:
# Example 1
query_exploration_icicle('rs_IsNewCredit', log_view, metric='avg_case_duration_seconds', details=False)

In [9]:
# Example 2
query_exploration_icicle('rs_SmallAmount', log_view, metric='avg_events_per_case', details=False)

In [10]:
# Example 3: details=True
query_exploration_icicle('rs_LoanOverThreshold', log_view, metric='avg_time_between_events', details=True)


Summary with Metrics:

- 🟡 10,084 cases (Initial Source → (CreditScore >= 600) ✅ → (RequestedAmount >= 10000) ✅) | Avg Time Between Events (s): 10h 21m
- 5,154 cases (Initial Source → (CreditScore >= 600) ✅ → (RequestedAmount >= 10000) ❌) | Avg Time Between Events (s): 9h 28m
- 10,020 cases (Initial Source → (CreditScore >= 600) ❌ → (RequestedAmount >= 10000) ✅) | Avg Time Between Events (s): 23h 7m
- 6,251 cases (Initial Source → (CreditScore >= 600) ❌ → (RequestedAmount >= 10000) ❌) | Avg Time Between Events (s): 23h 22m


In [11]:
# Example 4
query_breakdown_pie('rs_IsNewCredit', log_view, metric='avg_case_duration_seconds', details=False)

In [12]:
# Example 5
query_breakdown_pie('rs_SmallAmount', log_view, metric='avg_events_per_case', details=False)

In [13]:
# Example 6: details=True
query_breakdown_pie('rs_LoanOverThreshold', log_view, metric='avg_time_between_events', details=True)


Filter Paths:

- 🟡 10,084 cases ((CreditScore >= 600) ✅ → (RequestedAmount >= 10000) ✅) | Avg Time Between Events (s): 10h 10m
- 10,020 cases ((CreditScore >= 600) ❌ → (RequestedAmount >= 10000) ✅) | Avg Time Between Events (s): 20h 11m
